<a href="https://colab.research.google.com/github/sreedevrajendran/Used-Car-Price-Prediction/blob/main/Used_Car_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/dataset.csv')

# Display the first few rows of the DataFrame
print(df.head())

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
# Drop 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Clean 'Mileage' column
# Remove 'km/kg' or 'kmpl' units, convert to numeric, then fill NaNs with median
df['Mileage'] = df['Mileage'].astype(str).str.replace(r'(km/kg|kmpl)', '', regex=True).str.strip()
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')
df['Mileage'] = df['Mileage'].fillna(df['Mileage'].median())

# Clean 'Engine' column
# Remove 'CC' unit, convert to numeric, then fill NaNs with median
df['Engine'] = df['Engine'].astype(str).str.replace('CC', '', regex=False).str.strip()
df['Engine'] = pd.to_numeric(df['Engine'], errors='coerce')
df['Engine'] = df['Engine'].fillna(df['Engine'].median())

# Clean 'Power' column
# Remove 'bhp' unit, convert to numeric, then fill NaNs with median
df['Power'] = df['Power'].astype(str).str.replace('bhp', '', regex=False).str.strip()
df['Power'] = pd.to_numeric(df['Power'], errors='coerce')
df['Power'] = df['Power'].fillna(df['Power'].median())

# Handle 'New_Price' - remove 'Lakh', convert to numeric, then drop due to many NaNs
# Check if 'New_Price' column exists before processing/dropping
if 'New_Price' in df.columns:
    df['New_Price'] = df['New_Price'].astype(str).str.replace('Lakh', '', regex=False).str.strip()
    df['New_Price'] = pd.to_numeric(df['New_Price'], errors='coerce')
    # For simplicity, we'll drop New_Price due to the high number of missing values
    df = df.drop('New_Price', axis=1)

# Impute missing 'Seats' with the mode
df['Seats'] = df['Seats'].fillna(df['Seats'].mode()[0])

print(df.info())
print(df.describe())

In [ ]:
# Check for null values
print("Null values before removal:")
print(df.isnull().sum())

# Remove rows with any null values
df_cleaned = df.dropna()

# Check for null values after removal
print("\nNull values after removal:")
print(df_cleaned.isnull().sum())

print(f"\nOriginal DataFrame shape: {df.shape}")
print(f"Cleaned DataFrame shape: {df_cleaned.shape}")

df = df_cleaned.copy() # Update df to the cleaned version

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (x) and target (y)
# Drop 'Name' and 'Location' for now as they are categorical and require further encoding
x = df.drop(['Price', 'Name', 'Location', 'Fuel_Type', 'Transmission', 'Owner_Type'], axis=1)
y = df['Price']

# Split the data into training and testing sets
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

print(f"Shape of x_train: {xtrain.shape}")
print(f"Shape of x_test: {xtest.shape}")
print(f"Shape of y_train: {ytrain.shape}")
print(f"Shape of y_test: {ytest.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
model = LinearRegression()

In [ ]:
# Train the model
model.fit(xtrain, ytrain)

# Make predictions on the test set
y_pred = model.predict(xtest)

In [ ]:
# Create a sample DataFrame with features matching the training data (numerical only)
# You can modify these values to represent a car you're interested in.
sample_car_data = pd.DataFrame({
    'Year': [2018],
    'Kilometers_Driven': [50000],
    'Mileage': [18.0],
    'Engine': [1500.0],
    'Power': [100.0],
    'Seats': [5.0]
})

# Predict the price using the trained model
sample_prediction = model.predict(sample_car_data)

print(f"Predicted Price for the sample used car: {sample_prediction[0]:.2f} Lakhs")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot: Kilometers_Driven vs. Price
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Kilometers_Driven', y='Price', data=df)
plt.title('Kilometers Driven vs. Price')
plt.xlabel('Kilometers Driven')
plt.ylabel('Price (Lakhs)')
plt.grid(True)
plt.show()

In [ ]:
# Box plot: Fuel_Type vs. Price
plt.figure(figsize=(10, 6))
sns.boxplot(x='Fuel_Type', y='Price', data=df)
plt.title('Price Distribution by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Price (Lakhs)')
plt.grid(axis='y')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Get the counts for each fuel type
fuel_type_counts = df['Fuel_Type'].value_counts()

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(fuel_type_counts, labels=fuel_type_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Car Fuel Types')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# Calculate the R-squared score
r_squared = r2_score(ytest, y_pred)*100

print(f"R-squared Score: {r_squared:.2f}")